# Acoustic frequency response — 2D & 3D, all boundary conditions

End-to-end demo of the pyCAFE acoustic solvers on a duct-like cavity, in **3D (CHEXA8)** or **2D (CQUAD4)** — switch with the `DIM` flag below.

The model exercises every boundary condition and excitation of the dynamic system

$$\left(\mathbf K + j\omega\,\mathbf C - \omega^2 \mathbf M\right)\mathbf p = \mathbf Q + \mathbf V_n + \mathbf P$$

| Term | Physics | How it enters |
|---|---|---|
| $\mathbf V_n$ | vibrating piston, $\bar v_n$ along the **outward** normal | RHS vector $V_{n,a} = -j\rho_0\omega \int N_a \bar v_n\, d\Omega$ |
| $\mathbf C$ | impedance wall $p = \bar Z v_n$ | boundary matrix $C_{ab} = \int \rho_0 \bar A N_a N_b\, d\Omega$ |
| $\mathbf Q$ | point monopole $q_s\,\delta(\mathbf x - \mathbf x_s)$ | RHS vector $Q_a = j\rho_0\omega q_s N_a(\mathbf x_s)$ |
| $\bar p$ | pressure-release / prescribed pressure | DOF elimination + partitioning |
| rigid wall | $\bar v_n = 0$ | nothing (homogeneous Neumann) |

Both solution methods are run and compared:

- **direct**: solve $\mathbf D(\omega)\mathbf p = \mathbf F(\omega)$ at every frequency;
- **modal**: project on the rigid-cavity modes, solve for the participation factors $\phi_m$, reconstruct $\mathbf p = \mathbf\Phi\boldsymbol\phi$.

Sign convention: time dependence $e^{+j\omega t}$, $\bar v_n$ positive along the outward normal (a piston pushing **into** the fluid has $\bar v_n < 0$).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # run from examples/

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

from pycafe.boundary_condition import AcousticBC
from pycafe.core.prepare_acoustic_system import prepare_acoustic_system
from pycafe.solver.solver_helmholtz_1 import solve_helmholtz_frequency_sweep
from pycafe.solver.solver_modal_forced import (
    build_modal_basis,
    solve_modal_frequency_sweep,
)
from pycafe.build_matrices.bc_ops import expand_to_full

# ----------------------- user switches -----------------------
DIM = "3D"          # "3D" (CHEXA8 duct) or "2D" (CQUAD4 duct)

rho, c0 = 1.204, 343.0          # air
Lx, H = 1.0, 0.2                # duct length and height (and width in 3D)
frequencies = np.arange(20.0, 1000.0, 5.0)

v_piston = 0.01                 # piston velocity toward +x  [m/s]
zeta_wall = 2.0 - 1.0j          # normalized impedance of the treated wall
q_monopole = 5e-5               # monopole volume velocity [m^3/s]
NUM_MODES = 40                  # modal basis size (rule: f_m < 2 f_max)

## 1. Mesh

Structured mesh built in place (no Gmsh needed). Besides `elements`/`boundaries`, the builders return `walls`: every wall of the box as its own named face/edge group — used below to draw the boundary conditions and, in 3D, to reshape the solution onto the structured grid.

In [ ]:
def duct_mesh_2d(L, H, nx, ny):
    xs, ys = np.linspace(0, L, nx + 1), np.linspace(0, H, ny + 1)
    nodes = np.array([[x, y, 0.0] for y in ys for x in xs])
    tag = lambda i, j: j * (nx + 1) + i + 1
    quads = np.array([[tag(i, j), tag(i+1, j), tag(i+1, j+1), tag(i, j+1)]
                      for j in range(ny) for i in range(nx)])
    inlet  = np.array([[tag(0, j),  tag(0, j+1)]  for j in range(ny)])
    outlet = np.array([[tag(nx, j), tag(nx, j+1)] for j in range(ny)])
    top    = np.array([[tag(i, ny), tag(i+1, ny)] for i in range(nx)])
    bottom = np.array([[tag(i, 0),  tag(i+1, 0)]  for i in range(nx)])
    walls = {"inlet": inlet, "outlet": outlet, "top": top, "bottom": bottom}
    elements = {"Quadrilateral 4": quads,
                "Line 2": np.vstack([inlet, outlet, top])}
    boundaries = {n: sorted(set(w.ravel().tolist()))
                  for n, w in walls.items() if n != "bottom"}
    return nodes, elements, boundaries, walls, (nx, ny)


def duct_mesh_3d(L, H, W, nx, ny, nz):
    xs, ys, zs = (np.linspace(0, L, nx+1), np.linspace(0, H, ny+1),
                  np.linspace(0, W, nz+1))
    # z-major ordering: node index = k*(nx+1)*(ny+1) + j*(nx+1) + i
    nodes = np.array([[x, y, z] for z in zs for y in ys for x in xs])
    nxp, nyp = nx + 1, ny + 1
    tag = lambda i, j, k: k * nxp * nyp + j * nxp + i + 1
    hexes = np.array([
        [tag(i,j,k), tag(i+1,j,k), tag(i+1,j+1,k), tag(i,j+1,k),
         tag(i,j,k+1), tag(i+1,j,k+1), tag(i+1,j+1,k+1), tag(i,j+1,k+1)]
        for k in range(nz) for j in range(ny) for i in range(nx)])
    face_x = lambda i: np.array(
        [[tag(i,j,k), tag(i,j+1,k), tag(i,j+1,k+1), tag(i,j,k+1)]
         for k in range(nz) for j in range(ny)])
    face_y = lambda j: np.array(
        [[tag(i,j,k), tag(i+1,j,k), tag(i+1,j,k+1), tag(i,j,k+1)]
         for k in range(nz) for i in range(nx)])
    face_z = lambda k: np.array(
        [[tag(i,j,k), tag(i+1,j,k), tag(i+1,j+1,k), tag(i,j+1,k)]
         for j in range(ny) for i in range(nx)])
    walls = {"inlet": face_x(0), "outlet": face_x(nx),
             "top": face_y(ny), "bottom": face_y(0),
             "front": face_z(0), "back": face_z(nz)}
    elements = {"Hexahedron 8": hexes,
                "Quadrilateral 4": np.vstack([walls["inlet"], walls["outlet"],
                                              walls["top"]])}
    boundaries = {n: sorted(set(walls[n].ravel().tolist()))
                  for n in ("inlet", "outlet", "top")}
    return nodes, elements, boundaries, walls, (nx, ny, nz)


if DIM == "2D":
    # ~10 elements per wavelength at 1 kHz: h = c0/f/10 ~ 0.034 m
    nodes, elements, boundaries, walls, grid_n = duct_mesh_2d(Lx, H, nx=60, ny=12)
    x_src = [0.35 * Lx, 0.5 * H]                 # monopole position
    probe_xy = [0.8 * Lx, 0.5 * H]               # FRF probe
else:
    nodes, elements, boundaries, walls, grid_n = duct_mesh_3d(
        Lx, H, H, nx=40, ny=8, nz=8)
    x_src = [0.35 * Lx, 0.5 * H, 0.5 * H]
    probe_xy = [0.8 * Lx, 0.5 * H, 0.5 * H]

print(f"{DIM} mesh: {nodes.shape[0]} nodes, "
      f"{ {k: v.shape[0] for k, v in elements.items()} }")

## 2. Boundary conditions — all of them

- `inlet`: **normal velocity** (piston). Outward normal at $x=0$ points toward $-x$, so a piston moving toward $+x$ gives $\bar v_n = -v_\text{piston}$.
- `outlet`: **impedance** $\zeta = 2 - j$ (normalized by $\rho c$; use `(value, "abs")` for Pa·s/m, or a callable `Z(omega)` for a liner).
- `top`: **pressure release** $p = 0$ (Dirichlet, eliminated).
- a **point monopole** inside the domain.
- every unnamed wall: **rigid** (adds nothing).

> The modal path does not accept *non-zero* prescribed pressures (essential condition, not a load). $p=0$ is fine — it is eliminated before the modes are computed.

In [ ]:
bc = (AcousticBC()
      .add_velocity("inlet", -v_piston)       # piston (outward normal: -x)
      .add_impedance("outlet", zeta_wall)     # treated wall
      .add_pressure("top", 0.0)               # pressure release
      .add_monopole(q_monopole, position=x_src))

system = prepare_acoustic_system(
    nodes=nodes, elements=elements, boundaries=boundaries,
    rho=rho, c0=c0, bc=bc,
)

n_red = system["K_red"].shape[0]
print(f"DOFs: {nodes.shape[0]} total, {n_red} free "
      f"({nodes.shape[0] - n_red} eliminated by p=0)")
print(f"impedance operator frequency dependent: "
      f"{system['C_red_op'].is_frequency_dependent}")

## 3. Boundary conditions at a glance

Every wall drawn with the colour of its boundary condition, plus the monopole (★) and the FRF probe (●). This is exactly what `AcousticBC` above declares — nothing else is applied.

| colour | condition | term |
|---|---|---|
| red | normal velocity (piston) | $\mathbf V_n$ |
| orange | impedance $\zeta = 2-j$ | $j\omega\mathbf C$ |
| blue | prescribed $p=0$ | DOF elimination |
| grey | rigid wall | — |
| black ★ | monopole $q_s$ | $\mathbf Q$ |

In [ ]:
BC_STYLE = {                      # wall name -> (colour, label)
    "inlet":  ("crimson",   f"velocity $\\bar v_n = {-v_piston}$ m/s (piston)"),
    "outlet": ("darkorange", f"impedance $\\zeta = {zeta_wall}$"),
    "top":    ("royalblue", "pressure $p = 0$"),
}
RIGID = ("0.55", "rigid wall ($\\bar v_n = 0$)")

if DIM == "2D":
    fig, ax = plt.subplots(figsize=(9, 3))
    # domain fill + element grid (light)
    ax.tripcolor(nodes[:, 0], nodes[:, 1],
                 np.zeros(nodes.shape[0]), cmap="Greys", vmin=0, vmax=1,
                 alpha=0.06)
    seen = set()
    for name, segs in walls.items():
        color, label = BC_STYLE.get(name, RIGID)
        for seg in segs:
            xy = nodes[seg - 1]
            ax.plot(xy[:, 0], xy[:, 1], color=color, lw=4,
                    solid_capstyle="butt",
                    label=label if (color, label) not in seen else None)
            seen.add((color, label))
    ax.plot(*x_src, "k*", ms=16, label="monopole $q_s$")
    ax.plot(*probe_xy, "ko", ms=7, mfc="none", label="probe")
    ax.set_aspect("equal"); ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
    ax.set_title("Boundary conditions (2D)")
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=9)

else:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(projection="3d")
    handles = {}
    for name, faces in walls.items():
        color, label = BC_STYLE.get(name, RIGID)
        polys = [nodes[f - 1][:, :3] for f in faces]
        alpha = 0.55 if name in BC_STYLE else 0.10
        coll = Poly3DCollection(polys, facecolor=color, edgecolor="0.4",
                                lw=0.2, alpha=alpha)
        ax.add_collection3d(coll)
        handles[label] = Patch(facecolor=color, alpha=alpha)
    ax.scatter(*x_src, color="k", marker="*", s=220,
               depthshade=False, zorder=10)
    ax.scatter(*probe_xy, facecolor="none", edgecolor="k", s=70,
               depthshade=False, zorder=10)
    handles["monopole $q_s$"] = Line2D([], [], color="k", marker="*",
                                       ls="", ms=14)
    handles["probe"] = Line2D([], [], color="k", marker="o", ls="",
                              mfc="none", ms=8)
    ax.legend(handles.values(), handles.keys(),
              loc="center left", bbox_to_anchor=(1.05, 0.5), fontsize=9)
    ax.set_xlim(0, Lx); ax.set_ylim(0, H); ax.set_zlim(0, H)
    ax.set_box_aspect((Lx, H, H))
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_zlabel("z [m]")
    ax.set_title("Boundary conditions (3D)")
    ax.view_init(elev=22, azim=-60)

plt.tight_layout()
plt.show()

## 4. Direct solution

$\mathbf D(\omega)\,\mathbf p = \mathbf F(\omega)$ solved at each of the sweep frequencies. The boundary integrals were assembled once in `prepare_acoustic_system`; each frequency only re-scales them.

In [ ]:
%%time
P_direct = solve_helmholtz_frequency_sweep(
    system["K_red"].tocsr(), system["M_red"].tocsr(), system["C_red"],
    frequencies,
    system["pressure_nodes_red"], system["pressure_values"],
    velocity_operator=system["velocity_red_op"],
    impedance_operator=system["C_red_op"],
    source_operator=system["source_red_op"],
)

## 5. Modal solution

Basis = rigid-cavity modes of $(\mathbf K_a, \mathbf M_a)$, mass-normalized ($\tilde{\mathbf M} = \mathbf I$, $\tilde{\mathbf K} = \mathrm{diag}\,\omega_m^2$), **including the $\omega=0$ constant-pressure mode** (it carries the quasi-static response). The projected impedance $\tilde{\mathbf C} = \mathbf\Phi^T\mathbf C\mathbf\Phi$ is *not* diagonal — the impedance wall couples the rigid-cavity modes — so the small $m_a \times m_a$ complex system is solved per frequency: still tiny compared to the direct solve.

In [ ]:
%%time
basis = build_modal_basis(system["K_red"], system["M_red"], NUM_MODES)
omegas_m, Phi = basis
print(f"basis: {omegas_m.size} modes, f_m = 0 ... "
      f"{omegas_m[-1]/2/np.pi:.0f} Hz  (f_max = {frequencies[-1]:.0f} Hz, "
      f"rule f_m < 2 f_max)")

P_modal, modal = solve_modal_frequency_sweep(
    system["K_red"], system["M_red"], frequencies,
    num_modes=NUM_MODES, basis=basis,
    velocity_operator=system["velocity_red_op"],
    source_operator=system["source_red_op"],
    impedance_operator=system["C_red_op"],
    return_modal=True,
)

## 6. FRF at the probe — direct vs modal

In [ ]:
def closest_node(xy):
    d = np.linalg.norm(nodes[:, :len(xy)] - np.asarray(xy), axis=1)
    return int(np.argmin(d))

probe = closest_node(probe_xy)

p_dir_full = expand_to_full(P_direct, system["idx_free"],
                            system["p0_nodes"], nodes.shape[0])
p_mod_full = expand_to_full(P_modal, system["idx_free"],
                            system["p0_nodes"], nodes.shape[0])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True,
                               gridspec_kw={"height_ratios": [3, 1]})
ax1.semilogy(frequencies, np.abs(p_dir_full[probe]), "k-", lw=1.6,
             label="direct")
ax1.semilogy(frequencies, np.abs(p_mod_full[probe]), "r--", lw=1.2,
             label=f"modal ({NUM_MODES} modes)")
for w in omegas_m[1:]:
    f_m = w / (2 * np.pi)
    if f_m <= frequencies[-1]:
        ax1.axvline(f_m, color="0.85", zorder=0)
ax1.set_ylabel("|p| at probe [Pa]")
ax1.set_title(f"{DIM} duct — piston + impedance wall + p=0 top + monopole")
ax1.legend()

err = (np.abs(p_dir_full[probe] - p_mod_full[probe])
       / np.abs(p_dir_full[probe]).max())
ax2.semilogy(frequencies, err, "b-", lw=1.0)
ax2.set_xlabel("frequency [Hz]")
ax2.set_ylabel("modal trunc. err.")
plt.tight_layout()
plt.show()

print(f"max relative deviation modal vs direct: {err.max():.2e}")

Grey verticals = rigid-cavity natural frequencies (the basis). The response peaks are shifted/damped versions of them: the impedance wall moves the true resonances off the rigid-cavity ones — exactly why the modal basis needs headroom (rule $f_m < 2 f_\text{max}$).

## 7. Modal participations

Which mode carries the response: $|\phi_m(\omega)|$ map. A mode lights up when (a) the frequency is near $\omega_m$ and (b) the load shape overlaps the mode ($\tilde F_m = \boldsymbol\Phi_m^T\mathbf F \ne 0$).

In [ ]:
part = np.abs(modal["participations"])

fig, ax = plt.subplots(figsize=(9, 4))
pc = ax.pcolormesh(frequencies, np.arange(part.shape[0]),
                   np.log10(part + 1e-16), cmap="magma", shading="auto")
ax.set_xlabel("frequency [Hz]")
ax.set_ylabel("mode index m")
ax.set_title("log10 |participation factor|")
fig.colorbar(pc, ax=ax)
plt.tight_layout()
plt.show()

## 8. Pressure field at a chosen frequency

**2D**: field map on the section.

**3D**: full three-dimensional rendering — three orthogonal slice planes through the monopole position, drawn in a 3D axes and coloured by the field (the structured grid makes slicing exact, no interpolation).

In [ ]:
f_show = 550.0
i_show = int(np.argmin(np.abs(frequencies - f_show)))
p_show = p_dir_full[:, i_show]

if DIM == "2D":
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
    for ax, field, title, cmap in [
        (axes[0], np.real(p_show), "Re p", "RdBu_r"),
        (axes[1], np.abs(p_show), "|p|", "viridis"),
    ]:
        tp = ax.tripcolor(nodes[:, 0], nodes[:, 1], field,
                          shading="gouraud", cmap=cmap)
        ax.plot(*x_src, "k*", ms=12, label="monopole")
        ax.plot(nodes[probe, 0], nodes[probe, 1], "ko", ms=6, mfc="none",
                label="probe")
        ax.set_title(f"{title} at {frequencies[i_show]:.0f} Hz")
        ax.set_aspect("equal")
        fig.colorbar(tp, ax=ax)
    axes[0].legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.show()

else:
    # ---- reshape the solution onto the structured (k, j, i) grid ----
    nx, ny, nz = grid_n
    shape3 = (nz + 1, ny + 1, nx + 1)          # z-major node ordering
    X = nodes[:, 0].reshape(shape3)
    Y = nodes[:, 1].reshape(shape3)
    Z = nodes[:, 2].reshape(shape3)

    # slice indices through the monopole
    i_s = int(np.argmin(np.abs(X[0, 0, :] - x_src[0])))
    j_s = int(np.argmin(np.abs(Y[0, :, 0] - x_src[1])))
    k_s = int(np.argmin(np.abs(Z[:, 0, 0] - x_src[2])))

    for field, title, cmap in [
        (np.real(p_show).reshape(shape3), "Re p", "RdBu_r"),
        (np.abs(p_show).reshape(shape3), "|p|", "viridis"),
    ]:
        vmax = np.abs(field).max()
        vmin = -vmax if title == "Re p" else 0.0
        norm = plt.Normalize(vmin, vmax)
        cm = plt.get_cmap(cmap)

        fig = plt.figure(figsize=(10, 5.5))
        ax = fig.add_subplot(projection="3d")

        # x = x_src plane (j-k), y = y_src plane (i-k), z = z_src plane (i-j)
        ax.plot_surface(X[:, :, i_s], Y[:, :, i_s], Z[:, :, i_s],
                        facecolors=cm(norm(field[:, :, i_s])),
                        rstride=1, cstride=1, shade=False)
        ax.plot_surface(X[:, j_s, :], Y[:, j_s, :], Z[:, j_s, :],
                        facecolors=cm(norm(field[:, j_s, :])),
                        rstride=1, cstride=1, shade=False)
        ax.plot_surface(X[k_s, :, :], Y[k_s, :, :], Z[k_s, :, :],
                        facecolors=cm(norm(field[k_s, :, :])),
                        rstride=1, cstride=1, shade=False)

        # cavity wireframe for context
        for s, e in [((0,0,0),(Lx,0,0)), ((0,H,0),(Lx,H,0)),
                     ((0,0,H),(Lx,0,H)), ((0,H,H),(Lx,H,H)),
                     ((0,0,0),(0,H,0)), ((Lx,0,0),(Lx,H,0)),
                     ((0,0,H),(0,H,H)), ((Lx,0,H),(Lx,H,H)),
                     ((0,0,0),(0,0,H)), ((Lx,0,0),(Lx,0,H)),
                     ((0,H,0),(0,H,H)), ((Lx,H,0),(Lx,H,H))]:
            ax.plot(*zip(s, e), color="0.6", lw=0.8)

        ax.scatter(*x_src, color="k", marker="*", s=200, depthshade=False)
        ax.scatter(*probe_xy, facecolor="none", edgecolor="k", s=60,
                   depthshade=False)

        mappable = plt.cm.ScalarMappable(norm=norm, cmap=cm)
        fig.colorbar(mappable, ax=ax, shrink=0.7, label=f"{title} [Pa]")
        ax.set_xlim(0, Lx); ax.set_ylim(0, H); ax.set_zlim(0, H)
        ax.set_box_aspect((Lx, H, H))
        ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_zlabel("z [m]")
        ax.set_title(f"{title} at {frequencies[i_show]:.0f} Hz — "
                     "orthogonal slices through the monopole")
        ax.view_init(elev=22, azim=-60)
        plt.tight_layout()
        plt.show()

The $p=0$ top wall is visible as the field going to zero at $y = H$; the impedance outlet kills the reflection that a rigid wall would send back; the monopole shows up as the local hot spot on all three planes.

## 9. Sanity checks

- superposition: piston-only + monopole-only = both (linearity);
- the impedance wall dissipates: $|p|$ finite at every rigid-cavity resonance;
- modal ≈ direct within the truncation error seen above.

Things to try:

- `DIM = "2D"` — same script, CQUAD4 duct, edge-integrated BCs;
- a frequency-dependent liner: `add_impedance("outlet", lambda w: 2.0 + 300.0/w*1j)`;
- absolute impedance: `add_impedance("outlet", (830.0, "abs"))`  (Pa·s/m);
- move the monopole onto a node: `add_monopole(q, node=123)`;
- a distributed source: `add_distributed_source(q_density)`;
- Rayleigh damping in the modal solve: `rayleigh=(alpha, beta)`, or `modal_zeta=0.01`.